# Completions model spec comparison

Exploratory comparison of completions-model specifications (AR order, starts
lag, dummy set) on out-of-sample fit and residual whiteness

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error

ROOT = Path.cwd().resolve().parents[1]  # adjust if notebook moves
sys.path.append(str(ROOT))
from python.functions.bridge import parse_quarter

SPLIT_DATE = "2021-01-01"
HOUSEBUILDING_FILE = ROOT / "data/raw/starts/indicatorsofukhousebuilding.xlsx"

## Spec table

In [ ]:
SEASONAL = ["q1", "q2", "q3"]

# Dummy sets, named once, referenced by every spec that needs them.
OLD_DUM = ["d08Q3", "d20Q2", "d20Q3", "d23Q2", "d23Q3"]              # point GFC, regstd retained
NEW_DUM = ["gfc_2008_2009", "d20Q2", "d20Q3", "d20Q4"]                 # block GFC, regstd dropped
BLOCK_GFC_WITH_REGSTD = ["gfc_2008_2009", "d20Q2", "d20Q3", "d20Q4", "d23Q2", "d23Q3"]  # isolation run

AR1 = ["ln_C_lag1"]
AR4 = [f"ln_C_lag{i}" for i in range(1, 5)]

# name -> (AR terms, starts term, dummy set)
SPECS = {
    "Your model (AR1 + S_lag4 + OLD_DUM)":  (AR1, "ln_S_lag4", OLD_DUM),
    "NEW_DUM model (AR4 + S_lag1 + NEW_DUM)":    (AR4, "ln_S_lag1", NEW_DUM),
    "AR4 + S_lag1 + OLD_DUM":               (AR4, "ln_S_lag1", OLD_DUM),
    "AR4 + S_lag4 + OLD_DUM":               (AR4, "ln_S_lag4", OLD_DUM),
    "AR4 + S_lag4 + NEW_DUM":                (AR4, "ln_S_lag4", NEW_DUM),
    "AR4 + S_lag4 + block-GFC + regstd": (AR4, "ln_S_lag4", BLOCK_GFC_WITH_REGSTD),
}

## Load and prepare data

In [ ]:
def load_data(path: Path = HOUSEBUILDING_FILE) -> pd.DataFrame:
    raw = pd.read_excel(path, sheet_name="1b", header=5)
    df = raw.rename(columns={
        "Period": "period",
        "Started - Private Enterprise": "starts_private",
        "Completed - Private Enterprise": "completions_private",
    })[["period", "starts_private", "completions_private"]].copy()

    df["starts_private"] = pd.to_numeric(df["starts_private"], errors="coerce")
    df["completions_private"] = pd.to_numeric(df["completions_private"], errors="coerce")
    df["quarter"] = df["period"].apply(parse_quarter)
    df = df.dropna(subset=["quarter"]).set_index("quarter").sort_index().drop(columns="period")

    df["ln_C"] = np.log(df["completions_private"])
    df["ln_S"] = np.log(df["starts_private"])
    for L in range(1, 5):
        df[f"ln_C_lag{L}"] = df["ln_C"].shift(L)
    df["ln_S_lag1"] = df["ln_S"].shift(1)
    df["ln_S_lag4"] = df["ln_S"].shift(4)

    # get_dummies(drop_first=True) on quarter in {1,2,3,4} gives columns
    # q_2, q_3, q_4 in that order - i.e. calendar Q1 is the dropped baseline.
    q_dummies = pd.get_dummies(df.index.quarter, prefix="q", drop_first=True)
    q_dummies.columns = SEASONAL
    df = pd.concat([df, q_dummies.set_index(df.index)], axis=1)

    df["d08Q3"] = (df.index == "2008Q3").astype(int)
    df["d20Q2"] = (df.index == "2020Q2").astype(int)
    df["d20Q3"] = (df.index == "2020Q3").astype(int)
    df["d20Q4"] = (df.index == "2020Q4").astype(int)
    df["d23Q2"] = (df.index == "2023Q2").astype(int)
    df["d23Q3"] = (df.index == "2023Q3").astype(int)
    df["gfc_2008_2009"] = df.index.isin(pd.period_range("2008Q3", "2009Q4", freq="Q")).astype(int)

    df["date"] = df.index.to_timestamp()
    return df.dropna()


df = load_data()
train = df[df["date"] < SPLIT_DATE]
test = df[df["date"] >= SPLIT_DATE]
df.tail()

## Fit + evaluate helper

In [ ]:
def build_formula(ar_terms: list[str], starts_term: str, dummy_cols: list[str]) -> str:
    rhs = ar_terms + [starts_term] + SEASONAL + dummy_cols
    return "ln_C ~ " + " + ".join(rhs)


def fit_and_evaluate(train: pd.DataFrame, test: pd.DataFrame, ar_terms, starts_term, dummy_cols) -> dict:
    formula = build_formula(ar_terms, starts_term, dummy_cols)
    model = smf.ols(formula, data=train).fit()

    preds_level = np.exp(model.predict(test))
    actual_level = np.exp(test["ln_C"])
    lb_p4 = acorr_ljungbox(model.resid, lags=[4], return_df=True)["lb_pvalue"].iloc[0]

    return {
        "oos_rmse": np.sqrt(mean_squared_error(actual_level, preds_level)),
        "oos_mape_%": mean_absolute_percentage_error(actual_level, preds_level) * 100,
        "adj_r2": model.rsquared_adj,
        "bic": model.bic,
        "lb_pvalue_lag4": lb_p4,
    }

## Run comparison

In [ ]:
rows = {name: fit_and_evaluate(train, test, *spec) for name, spec in SPECS.items()}
results = pd.DataFrame(rows).T
results.index.name = "spec"
results.style.format("{:.4f}")

## Read

- Lower BIC / OOS MAPE / OOS RMSE is better.
- `lb_pvalue_lag4` > ~0.05 means residuals pass the whiteness check at lag 4
  (no leftover serial correlation) — tNEW_DUM is the diagnostic that ruled out
  the AR(1) and lag-1-starts specs.
- Compare `AR4 + S_lag4 + NEW_DUM` against `AR4 + S_lag4 + block-GFC + regstd`
  directly: if the isolation spec keeps the clean Ljung-Box p-value, the
  block-GFC dummy is what's doing the work and regstd can be reinstated.
  If it fails, dropping regstd was load-bearing.

## Regstd isolation (full-sample, in-sample)

The train/test comparison above cannot identify `d23Q2`/`d23Q3` because both
quarters sit in the test window. TNEW_DUM refits on the **full sample** so the
coefficients are identified, and answers the question three ways:

1. **Individual significance** — t-stats/p-values on `d23Q2`, `d23Q3`.
2. **Joint significance** — F-test of `d23Q2 = d23Q3 = 0`, which is the
   right test since the two dummies mark one regulatory transition, not two
   independent shocks.
3. **Do residuals still pass whiteness without them** — Ljung-Box on both
   fits. If dropping regstd leaves LB clean, exclusion is defensible on
   parsimony; if dropping it breaks LB, regstd was load-bearing and the
   earlier `NEW_DUM` result was picking up the block-GFC change only.

Note the diagnostics here are in-sample on the full sample, so they are not
comparable to the OOS metrics in the table above — tNEW_DUM cell is answering a
specification question, not re-running the horse race.

In [ ]:
# Full-sample fits: with and without the regstd point dummies.
# Both use the block-GFC dummy set, so the ONLY difference is regstd.
REGSTD = ["d23Q2", "d23Q3"]

fit_with = smf.ols(build_formula(AR4, "ln_S_lag4", BLOCK_GFC_WITH_REGSTD), data=df).fit()
fit_without = smf.ols(build_formula(AR4, "ln_S_lag4", NEW_DUM), data=df).fit()

# Individual significance of the regstd dummies
regstd_tbl = pd.DataFrame({
    "coef": fit_with.params[REGSTD],
    "std_err": fit_with.bse[REGSTD],
    "t": fit_with.tvalues[REGSTD],
    "p": fit_with.pvalues[REGSTD],
})
print("Individual significance (full-sample fit, regstd retained)")
print(regstd_tbl.round(4))
print()

# Joint significance: d23Q2 = d23Q3 = 0
joint = fit_with.f_test("d23Q2 = 0, d23Q3 = 0")
print(f"Joint F-test (d23Q2 = d23Q3 = 0): F = {float(joint.fvalue):.4f}, p = {float(joint.pvalue):.4f}")
print()

# Does whiteness survive dropping them?
compare = pd.DataFrame({
    "with_regstd": {
        "lb_pvalue_lag4": acorr_ljungbox(fit_with.resid, lags=[4], return_df=True)["lb_pvalue"].iloc[0],
        "adj_r2": fit_with.rsquared_adj,
        "bic": fit_with.bic,
        "n_obs": int(fit_with.nobs),
    },
    "without_regstd": {
        "lb_pvalue_lag4": acorr_ljungbox(fit_without.resid, lags=[4], return_df=True)["lb_pvalue"].iloc[0],
        "adj_r2": fit_without.rsquared_adj,
        "bic": fit_without.bic,
        "n_obs": int(fit_without.nobs),
    },
})
print("Whiteness / fit comparison (full-sample, in-sample)")
print(compare.round(4))

### How to read tNEW_DUM cell

- **Joint p > 0.05 and LB stays clean without regstd** → drop them. Exclusion
  is justified on parsimony and the earlier `NEW_DUM` advantage was the
  block-GFC dummy doing the work. Leave `DEFAULT_DUMMY_COLS` as-is.
- **Joint p < 0.05** → reinstate them: add `"d23Q2", "d23Q3"` to
  `DEFAULT_DUMMY_COLS` in `recursive_completions_forecast.py`. Nothing else
  in the recursion changes — future quarters zero them either way.
- **Joint p > 0.05 but LB degrades badly without them** → they are absorbing
  something the joint test is too underpowered to detect on two observations.
  Retain, and say so explicitly as a judgement call rather than a test result.

One caveat worth stating in the write-up either way: point dummies on two
observations are a low-power test by construction, so a non-rejection here is
weak evidence of absence, not evidence the regstd transition had no effect on
completions.